# 笔记本 06 — 制度检测

**阶段 2 · 策略模块 (3 / 4)**

---

## 🎯 学习目标

| # | 目标 |
|---|------|
| 1 | 理解*为什么*制度检测是制度自适应机器人的核心 |
| 2 | 使用 EMA 交叉 + 波动率将市场分类为**牛市 / 震荡 / 熊市** |
| 3 | 实现**反震荡确认过滤器**以减少虚假制度切换 |
| 4 | 用颜色编码的价格图表可视化制度历史 |
| 5 | 将手写代码与生产环境的 `classify_regime_history()` 对比 |

### 前置要求
- NB02（EMA）
- NB04 & NB05（动量 + 均值回归 — 理解*为什么*制度很重要）

In [ ]:
# ── 初始化 ─────────────────────────────────────────────────
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = str(pathlib.Path.cwd().resolve().parents[1])
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates

plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True})
print("✅ 导入完毕  |  项目根目录:", ROOT)

---
## 1 · 为什么需要制度检测？

没有单一策略能在所有市场条件下有效：

| 制度 | 最佳策略 | 最差策略 |
|------|----------|----------|
| **牛市** 📈 | 动量（顺势而为） | 均值回归（逆势） |
| **震荡** ↔️ | 均值回归（在极端处反转） | 动量（被来回震荡） |
| **熊市** 📉 | 防御性 / 现金 | 所有策略（尤其是动量） |

我们的机器人使用制度检测来动态**重新分配**四个子策略的权重。制度检测器是管线中**最重要的单一决策**。

### 检测逻辑（来自 `config/strategy_params.yaml`）

```yaml
regime:
  ema_fast_period: 20
  ema_slow_period: 50
  volatility_lookback: 14
  volatility_threshold_multiplier: 1.5
  confirmation_periods: 2
```

---
## 2 · 具有多种制度的合成价格数据

In [ ]:
np.random.seed(42)
DAYS = 300
dates = pd.date_range(end=pd.Timestamp.now().normalize(), periods=DAYS, freq="D")

# 构造具有明确制度阶段的价格序列
drift = np.zeros(DAYS)
vol   = np.zeros(DAYS)
true_regime = []

# 阶段 1: 牛市 (第 0–80 天)
drift[0:80]   = 0.003;  vol[0:80]   = 0.015
true_regime += ["bull"] * 80
# 阶段 2: 震荡 (第 80–160 天)
drift[80:160]  = 0.000;  vol[80:160]  = 0.020
true_regime += ["ranging"] * 80
# 阶段 3: 熊市 (第 160–220 天)
drift[160:220] = -0.004; vol[160:220] = 0.035
true_regime += ["bear"] * 60
# 阶段 4: 恢复牛市 (第 220–300 天)
drift[220:300] = 0.004;  vol[220:300] = 0.018
true_regime += ["bull"] * 80

log_returns = np.array([np.random.normal(d, v) for d, v in zip(drift, vol)])
prices = pd.Series(60000 * np.exp(np.cumsum(log_returns)), index=dates, name="BTCUSDT")

true_regime_series = pd.Series(true_regime, index=dates, name="true_regime")

print(f"价格范围: ${prices.min():,.0f} – ${prices.max():,.0f}")
print(f"真实制度: {dict(true_regime_series.value_counts())}")

---
## 3 · 第一步：EMA 交叉

第一个信号是**快速 EMA**（20 天）和**慢速 EMA**（50 天）之间的关系：

- **快 > 慢** → 上升趋势偏向（牛市）
- **快 < 慢** → 下降趋势偏向（熊市 / 震荡）

In [ ]:
EMA_FAST = 20   # 来自 strategy_params.yaml
EMA_SLOW = 50

ema_fast = prices.ewm(span=EMA_FAST, adjust=False).mean()
ema_slow = prices.ewm(span=EMA_SLOW, adjust=False).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(prices.index, prices, label="价格", lw=1, alpha=0.7)
ax.plot(ema_fast.index, ema_fast, label=f"EMA-{EMA_FAST}", lw=1.5)
ax.plot(ema_slow.index, ema_slow, label=f"EMA-{EMA_SLOW}", lw=1.5)

# 着色：牛市（快 > 慢）vs 熊市（快 < 慢）
bull_mask = ema_fast > ema_slow
ax.fill_between(prices.index, prices.min() * 0.9, prices.max() * 1.1,
                where=bull_mask, alpha=0.05, color="green", label="快 > 慢")
ax.fill_between(prices.index, prices.min() * 0.9, prices.max() * 1.1,
                where=~bull_mask, alpha=0.05, color="red", label="快 < 慢")
ax.legend(fontsize=8)
ax.set_title("EMA 交叉信号")
ax.set_ylabel("价格")
plt.tight_layout()
plt.show()

---
## 4 · 第二步：波动率阈值

高波动率会覆盖 EMA 的牛市判定 → 强制分类为**熊市**。

$$
\text{vol\_threshold} = \sigma_{\text{baseline}}(60\text{d}) \times 1.5
$$

如果当前 14 天实现波动率超过此阈值，即使 EMA 看涨，制度也会翻转为**熊市**。

In [ ]:
VOL_LOOKBACK = 14
VOL_BASELINE = 60
VOL_MULTIPLIER = 1.5

returns = prices.pct_change()
rolling_vol = returns.rolling(VOL_LOOKBACK).std(ddof=0)
baseline_vol = returns.rolling(VOL_BASELINE).std(ddof=0)
vol_threshold = baseline_vol * VOL_MULTIPLIER

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(rolling_vol.index, rolling_vol, label=f"{VOL_LOOKBACK}天 实现波动率", lw=1.2)
ax.plot(vol_threshold.index, vol_threshold, label=f"阈值 ({VOL_MULTIPLIER}× 基线)",
        lw=1.2, ls="--", color="red")
high_vol = rolling_vol > vol_threshold
ax.fill_between(rolling_vol.index, 0, rolling_vol.max() * 1.2,
                where=high_vol, alpha=0.15, color="red", label="波动率偏高")
ax.set_ylabel("波动率")
ax.set_title("波动率 vs 自适应阈值")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## 5 · 第三步：组合 → 基础制度

`detect_regime()` 函数将 EMA + 波动率组合成单一分类：

In [ ]:
def detect_regime(price, ema_f, ema_s, vol, vol_thresh):
    """将单根 K 线分类为 牛市 / 震荡 / 熊市。"""
    # 强牛市：价格在两条 EMA 之上 且 低波动
    if price > ema_f > ema_s and vol <= vol_thresh:
        return "bull"
    # 强熊市：价格在两条 EMA 之下
    if price < ema_f < ema_s:
        return "bear"
    # 高波动覆盖 → 熊市
    if vol > vol_thresh:
        return "bear"
    # EMA 看涨排列 → 牛市
    if ema_f > ema_s:
        return "bull"
    # 默认
    return "ranging"

# 逐根 K 线分类
base_regimes = []
for ts in prices.index:
    if any(pd.isna(x) for x in [ema_fast[ts], ema_slow[ts], rolling_vol[ts], vol_threshold[ts]]):
        base_regimes.append("unknown")
    else:
        base_regimes.append(detect_regime(
            float(prices[ts]), float(ema_fast[ts]), float(ema_slow[ts]),
            float(rolling_vol[ts]), float(vol_threshold[ts]),
        ))

base_regime_series = pd.Series(base_regimes, index=prices.index)
print("基础制度计数:")
print(base_regime_series.value_counts())

---
## 6 · 第四步：反震荡确认过滤器

基础制度频繁翻转 — 这导致**whipsaw**（快速来回切换，触发不必要的再平衡）。**确认过滤器**要求一个制度持续 `confirmation_periods` 根 K 线后才成为**活跃制度**。

```
天:     1    2    3    4    5    6    7    8
基础:   牛市 牛市 熊市 熊市 熊市 牛市 牛市 牛市
活跃:   牛市 牛市 牛市 牛市 熊市 熊市 熊市 牛市  ← 确认延迟
```

当 `confirmation_periods=2` 时，新制度必须连续出现 2 根 K 线后，活跃制度才会切换。

In [ ]:
CONFIRMATION = 2  # 来自 strategy_params.yaml

def apply_confirmation(base_regimes: list[str], confirmation_periods: int) -> list[str]:
    """反震荡：仅在连续 N 次确认后才改变活跃制度。"""
    active_regimes = []
    last_active = "unknown"
    streak_regime = "unknown"
    streak_count  = 0
    
    for regime in base_regimes:
        if regime == "unknown":
            active_regimes.append(last_active)
            continue
        
        # 追踪连续计数
        if regime == streak_regime:
            streak_count += 1
        else:
            streak_regime = regime
            streak_count = 1
        
        # 第一个非 unknown
        if last_active == "unknown":
            last_active = regime
        elif regime != last_active and streak_count >= max(1, confirmation_periods):
            last_active = regime
        
        active_regimes.append(last_active)
    
    return active_regimes

active_regimes = apply_confirmation(base_regimes, CONFIRMATION)
active_regime_series = pd.Series(active_regimes, index=prices.index)

# 比较基础 vs 活跃
switches_base   = (base_regime_series != base_regime_series.shift(1)).sum()
switches_active = (active_regime_series != active_regime_series.shift(1)).sum()
print(f"基础制度切换次数:   {switches_base}")
print(f"活跃制度切换次数:   {switches_active}")
print(f"减少:               少了 {switches_base - switches_active} 次 ({(1 - switches_active/switches_base)*100:.0f}%)")

---
## 7 · 可视化：制度着色价格图

In [ ]:
REGIME_COLOURS = {"bull": "#2ecc71", "ranging": "#f39c12", "bear": "#e74c3c", "unknown": "#95a5a6"}
REGIME_NAMES   = {"bull": "牛市", "ranging": "震荡", "bear": "熊市", "unknown": "未知"}

def plot_regime_chart(prices, regimes, title):
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(prices.index, prices, color="black", lw=1, alpha=0.7)
    
    # 按制度着色背景
    prev_regime = regimes.iloc[0]
    start = prices.index[0]
    for i in range(1, len(regimes)):
        if regimes.iloc[i] != prev_regime or i == len(regimes) - 1:
            end = prices.index[i]
            ax.axvspan(start, end, alpha=0.2, color=REGIME_COLOURS.get(prev_regime, "gray"))
            start = end
            prev_regime = regimes.iloc[i]
    
    ax.set_title(title)
    ax.set_ylabel("价格")
    patches = [mpatches.Patch(color=c, alpha=0.3, label=REGIME_NAMES[r]) for r, c in REGIME_COLOURS.items() if r != "unknown"]
    ax.legend(handles=patches, loc="upper left")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    plt.tight_layout()
    plt.show()

plot_regime_chart(prices, active_regime_series, "检测到的制度（含确认过滤器）")

In [ ]:
# 并排对比：基础 vs 活跃 vs 真实
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, series, title in [
    (axes[0], true_regime_series, "真实制度"),
    (axes[1], base_regime_series, "基础制度（无确认）"),
    (axes[2], active_regime_series, "活跃制度（确认=2）"),
]:
    ax.plot(prices.index, prices, color="black", lw=0.8, alpha=0.6)
    prev_r = series.iloc[0]
    start = prices.index[0]
    for i in range(1, len(series)):
        if series.iloc[i] != prev_r or i == len(series) - 1:
            end = prices.index[i]
            ax.axvspan(start, end, alpha=0.25, color=REGIME_COLOURS.get(prev_r, "gray"))
            start = end
            prev_r = series.iloc[i]
    ax.set_title(title)
    ax.set_ylabel("价格")

patches = [mpatches.Patch(color=c, alpha=0.3, label=REGIME_NAMES[r]) for r, c in REGIME_COLOURS.items() if r != "unknown"]
axes[0].legend(handles=patches, loc="upper left", fontsize=8)
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.tight_layout()
plt.show()

---
## 8 · 制度准确率分析

In [ ]:
# 仅比较两者都已知的部分
mask = (active_regime_series != "unknown") & (true_regime_series != "unknown")
compare = pd.DataFrame({
    "true": true_regime_series[mask],
    "detected": active_regime_series[mask],
})
compare["match"] = compare["true"] == compare["detected"]
accuracy = compare["match"].mean()
print(f"整体准确率: {accuracy:.1%}\n")

# 分制度准确率
for regime in ["bull", "ranging", "bear"]:
    subset = compare[compare["true"] == regime]
    if len(subset) > 0:
        acc = subset["match"].mean()
        print(f"  {REGIME_NAMES[regime]:8s}: {acc:.1%}  ({len(subset)} 根 K 线)")

# 混淆矩阵
print("\n混淆矩阵:")
print(pd.crosstab(compare["true"], compare["detected"], margins=True))

---
## 9 · 生产代码：`classify_regime_history()`

In [ ]:
from bot.strategy.regime_detector import classify_regime_history, detect_regime as prod_detect

prod_df = classify_regime_history(
    prices,
    ema_fast_period=20,
    ema_slow_period=50,
    volatility_lookback=14,
    volatility_baseline_period=60,
    volatility_threshold_multiplier=1.5,
    confirmation_periods=2,
)

print("生产输出列:", prod_df.columns.tolist())
prod_df[["price", "ema_fast", "ema_slow", "volatility", "base_regime", "active_regime"]].tail(5)

In [ ]:
# 验证手写代码与生产代码一致
prod_active = prod_df["active_regime"]
match_rate = (prod_active == active_regime_series).mean()
print(f"手写 vs 生产 一致率: {match_rate:.1%}")

# 展示生产制度图表
plot_regime_chart(prices, prod_active, "生产环境制度检测")

---
## 10 · 制度对策略选择的影响

这是通往 NB07（集成策略）的桥梁。制度决定每个子策略获得的**权重**：

```python
# 来自 bot/strategy/ensemble.py
_REGIME_WEIGHTS = {
    "bull":    {"momentum": 0.5, "mean_reversion": 0.1, "pairs": 0.2, "sector": 0.2},
    "ranging": {"momentum": 0.2, "mean_reversion": 0.4, "pairs": 0.3, "sector": 0.1},
    "bear":    {"momentum": 0.1, "mean_reversion": 0.3, "pairs": 0.3, "sector": 0.3},
}
```

In [ ]:
REGIME_WEIGHTS = {
    "bull":    {"momentum": 0.5, "mean_reversion": 0.1, "pairs": 0.2, "sector": 0.2},
    "ranging": {"momentum": 0.2, "mean_reversion": 0.4, "pairs": 0.3, "sector": 0.1},
    "bear":    {"momentum": 0.1, "mean_reversion": 0.3, "pairs": 0.3, "sector": 0.3},
}

weight_df = pd.DataFrame(REGIME_WEIGHTS).T
weight_df.index = [REGIME_NAMES[r] for r in weight_df.index]
weight_df.columns = ["动量", "均值回归", "配对", "板块轮动"]
weight_df.plot.bar(figsize=(10, 5), rot=0, width=0.7,
                   color=["#3498db", "#e74c3c", "#f39c12", "#2ecc71"])
plt.title("各制度下的子策略权重")
plt.ylabel("权重")
plt.xlabel("制度")
plt.legend(title="策略")
plt.tight_layout()
plt.show()

---
## 11 · 敏感性分析：确认期数

`confirmation_periods` 如何影响切换频率？

In [ ]:
results = []
for cp in range(1, 8):
    active = apply_confirmation(base_regimes, cp)
    active_s = pd.Series(active, index=prices.index)
    n_switches = (active_s != active_s.shift(1)).sum()
    # 与真实制度的准确率
    m = (active_s != "unknown") & (true_regime_series != "unknown")
    acc = (active_s[m] == true_regime_series[m]).mean()
    results.append({"confirmation": cp, "switches": n_switches, "accuracy": acc})

res_df = pd.DataFrame(results).set_index("confirmation")

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.bar(res_df.index, res_df["switches"], alpha=0.6, color="#3498db", label="切换次数")
ax2.plot(res_df.index, res_df["accuracy"], color="#e74c3c", marker="o", lw=2, label="准确率")
ax1.set_xlabel("确认期数")
ax1.set_ylabel("切换次数", color="#3498db")
ax2.set_ylabel("vs 真实制度的准确率", color="#e74c3c")
ax1.set_title("确认期数：稳定性 vs 准确率权衡")
fig.legend(loc="upper right", bbox_to_anchor=(0.88, 0.88))
plt.tight_layout()
plt.show()

res_df.style.format({"accuracy": "{:.1%}"})

---
## 12 · 关键要点

| 概念 | 详情 |
|------|------|
| **EMA 交叉** | 快(20) vs 慢(50) — 主要趋势信号 |
| **波动率覆盖** | 14天波动率 > 1.5× 基线(60天) → 熊市 |
| **确认过滤器** | 必须持续 2 根 K 线才能切换活跃制度 |
| **三种制度** | 牛市 / 震荡 / 熊市 |
| **下游影响** | 制度 → 4 个子策略的权重分配 |

### 制度检测管线

```
价格序列
  │
  ├── EMA 快线 (20) & EMA 慢线 (50)
  ├── 14天 实现波动率
  ├── 60天 基线波动率 × 1.5 阈值
  │
  ├── detect_regime() → 基础制度（逐根 K 线）
  │
  └── apply_confirmation(2) → 活跃制度（反震荡）
```

---
## 🔬 练习

1. **EMA 周期扫描：** 尝试快/慢组合 (10, 30)、(20, 50) 和 (30, 100)。哪个在合成数据上的准确率最高？

2. **波动率乘数：** 将 `vol_threshold_multiplier` 从 1.0 到 2.5 每隔 0.25 变化。在什么水平下，熊市制度几乎消失？

3. **隐马尔可夫模型：** 用 3 状态隐马尔可夫模型（使用 `hmmlearn`）替代 EMA 交叉。它是否以更多误报为代价更早检测到制度变化？

4. **制度转移矩阵：** 绘制转移矩阵：给定当前活跃制度，下一个制度的概率是多少？转移是否对称？

---
## ✅ 知识检查

1. 为什么机器人使用*两条* EMA 而不是一条？
2. 如果在牛市趋势中波动率飙升（价格 > EMA_快 > EMA_慢）会发生什么？
3. 确认过滤器如何减少震荡？权衡是什么？
4. 动量在哪种制度下获得最高权重？为什么？
5. 你如何添加第四种制度（例如"高波动牛市"）？

---
## 🔗 下一步

**[NB07 — 集成策略与情绪 →](07_集成策略与情绪.ipynb)**

我们将把所有内容整合在一起：使用制度加权混合来组合动量、均值回归、配对和板块轮动，并叠加来自恐惧与贪婪指数的情绪。